# Klein unified training
This notebook controls the same sync, discovery, queue, and report code used by `/run-unified-training`. Choose one explicit action, then **Run All**. The default `status` action only reads durable state.

In [ ]:
SOURCE_REVISION = "30624604fb6fb39ef85dfe25ea25a6f6c75cdba6"
REPO_ID = "daverave/Personal"
RUN_ID = "e372619d-f4dc-4f4b-a2ba-b0df127e24fc"  # editable, never guessed
DATASET_ROOT = "/storage/datasets"
COMFYUI_ROOT = "/storage/ComfyUI"
LORA_ROOT = "/storage/ComfyUI/models/loras"
WORK_ROOT = "/storage/automation/unified"
WORKER_ID, WORKER_COUNT = 0, 1
RANKS = [1]
MODEL_IDS = []  # empty means every model with a latest completed pointer
ORDER = []      # normalized folder or catalog names; unlisted jobs stay alphabetical
ACTION = "status"  # status | sync | discover | run | refresh
REFRESH_JOBS = []  # [(evaluation_json, samples_root, run_id), ...]

In [ ]:
import pathlib, shutil, subprocess, sys
checkout = pathlib.Path('/tmp/ai-toolkit-perceptual-pinned')
if checkout.exists(): shutil.rmtree(checkout)
subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/Explyy/ai-toolkit-perceptual.git', str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', SOURCE_REVISION], check=True)
assert subprocess.run(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip() == SOURCE_REVISION
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'huggingface_hub>=0.27,<2', 'PyYAML>=6,<7', 'Pillow>=12,<13', 'numpy>=1.26,<3'], check=True)
sys.path.insert(0, str(checkout))

In [ ]:
import getpass, json, os, yaml
token = os.environ.get('HF_TOKEN') or getpass.getpass('HF private repository token: ')
os.environ['HF_TOKEN'] = token
os.environ['HF_REPO_ID'] = REPO_ID
os.environ['DATASETS_FOLDER'] = DATASET_ROOT
config = {
 'schema_version': 1, 'dataset_root': DATASET_ROOT, 'comfyui_root': COMFYUI_ROOT, 'loras_root': LORA_ROOT, 'work_root': WORK_ROOT,
 'hub': {'repo_id': REPO_ID, 'repo_type': 'dataset', 'token_env': 'HF_TOKEN'},
 'worker': {'id': WORKER_ID, 'count': WORKER_COUNT},
 'sync': {'catalog_prefix': 'training-backups', 'results_prefix': 'training-results'},
 'discovery': {'quiet_seconds': 60, 'target_exposures': 126, 'ledger_path': 'training-automation/workflow-ledger.json', 'retry_incomplete': False, 'order': ORDER},
 'queue': {'trainer_yaml': str(checkout / 'config/examples/klein_automation/trainer-subject-likeness-masked-klein-9b-v2.yaml'), 'repo_root': str(checkout), 'output_root': '/storage/output', 'trigger_word': 'Owhx'},
 'checkpoint_policy': {'save_every': 100, 'max_local_step_saves': 5},
 'evaluation': {'enabled': True, 'require_identity_available': True, 'reference_provenance': 'training-set', 'face_backend': 'training_automation.backends:InsightFaceCPUBackend', 'face_backend_options': {'model_dir': '/opt/training-automation-models/insightface/models/buffalo_l'}, 'reference_identity_filter': {'single_face_only': True, 'minimum_valid_count': 3, 'minimum_valid_fraction': 0.5}, 'landmark_backend': 'training_automation.backends:UltralyticsPoseCPUBackend', 'landmark_backend_options': {'model_path': '/opt/training-automation-models/ultralytics/yolo11n-pose.pt', 'expected_sha256': '869e83fcdffdc7371fa4e34cd8e51c838cc729571d1635e5141e3075e9319dc0'}},
 'archive': {'remote_prefix': 'training-archives'}
}
config_path = pathlib.Path(WORK_ROOT) / 'notebook-config.yaml'
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

In [ ]:
from training_automation.backup import HuggingFaceBackupClient
from training_automation.results import publish_refreshed_report
from training_automation.state import read_json
from training_automation.sync import sync_latest_loras
from training_automation.unified import run_unified_workflow
client = HuggingFaceBackupClient(token)
if ACTION == 'status':
    result = read_json(pathlib.Path(WORK_ROOT) / 'supervisor-state.json', {'schema_version': 1, 'status': 'not-started'})
elif ACTION == 'sync':
    _, revision = client.read_remote_file(REPO_ID, 'dataset', 'training-backups/catalog.json')
    result = sync_latest_loras(client=client, repo_id=REPO_ID, repo_type='dataset', source_revision=revision, loras_root=pathlib.Path(LORA_ROOT), work_dir=pathlib.Path(WORK_ROOT) / 'notebook-sync', ranks=RANKS, model_ids=MODEL_IDS)
elif ACTION in {'discover', 'run'}:
    result = run_unified_workflow(config_path, dry_run=(ACTION == 'discover'))
elif ACTION == 'refresh':
    result = [publish_refreshed_report(client=client, repo_id=REPO_ID, repo_type='dataset', run_id=run_id, report_path=pathlib.Path(report), sample_root=pathlib.Path(samples), work_dir=pathlib.Path(WORK_ROOT) / 'report-refresh' / pathlib.Path(report).parent.name) for report, samples, run_id in REFRESH_JOBS]
else:
    raise ValueError('ACTION must be status, sync, discover, run, or refresh')
print(json.dumps(result, indent=2, sort_keys=True))